# Old vs new cellina metrics

Old numbers: `results/loo_summary_{dataset_name}_DEG_50_v3.csv`. New numbers: JSONs in `DATA_ROOT/datasets/{dataset_name}/correlations` (read as in `ood_plots.ipynb`).

- edge: `cellina` (csv) vs `cellina-cr-cf` (json)
- node: `cellina-pert` (csv) vs `cellina-cr-cf-node` (json)

In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)

In [2]:
dataset_name = "crc"
DATA_ROOT = '/data2/a330d'  # JSON_BASE_PATH
SUMMARY_CSV = f'../loo_summary_{dataset_name}_DEG_50_v5.csv'

# old model name (csv) -> new model name (json)
PAIRS = {
    'cellina': 'cellina-cr-cf',
    'cellina-pert': 'cellina-cr-cf-node',
}
METRICS = ['pearson', 'direction_match_k', 'edistance_pca_log', 'rmse']
KEYS = ['sid', 'holdout_celltype']

## Load

In [3]:
df_old = pd.read_csv(SUMMARY_CSV)
df_old = df_old[df_old['model_name'].isin(PAIRS.keys())]
df_old.groupby('model_name').size()

model_name
cellina         30
cellina-pert    30
dtype: int64

In [4]:
corr_dir = os.path.join(DATA_ROOT, f"datasets/{dataset_name}/correlations")
pattern = os.path.join(corr_dir, "*.json")
files = sorted(glob.glob(pattern))

rows = []
for fp in files:
    name = os.path.basename(fp)
    core = name[len("crc_"):-len(".json")] if dataset_name == "crc" else name[:-len(".json")]
    parts = core.split("_")
    sid = parts[0]
    model_name = parts[1]
    holdout_celltype = "_".join(parts[2:])
    if model_name not in PAIRS.values():
        continue
    with open(fp, "r") as f:
        data = json.load(f)
    rows.append({
        "sid": f"crc_{sid}" if dataset_name == "crc" else sid,
        "model_name": model_name,
        "holdout_celltype": holdout_celltype,
        **{m: data.get(m) for m in METRICS},
    })

df_new = pd.DataFrame(rows)
df_new.groupby('model_name').size()

model_name
cellina-cr-cf         6
cellina-cr-cf-node    6
dtype: int64

## Side-by-side per run

Rows: runs present in both csv and json. Columns: metric → old | new | Δ (new − old).

In [5]:
def side_by_side(old_name, new_name):
    old = df_old[df_old['model_name'] == old_name].set_index(KEYS)[METRICS]
    new = df_new[df_new['model_name'] == new_name].set_index(KEYS)[METRICS]
    old, new = old.align(new, join='inner')
    out = pd.concat({m: pd.DataFrame({f'old ({old_name})': old[m],
                                      f'new ({new_name})': new[m],
                                      'Δ': np.round(new[m] - old[m], 2)}) for m in METRICS}, axis=1)
    return out

comparisons = {old_name: side_by_side(old_name, new_name) for old_name, new_name in PAIRS.items()}

In [6]:
comparisons['cellina'].round(3)

pearson                          direction_match_k                          edistance_pca_log                                    rmse                         
                         old (cellina) new (cellina-cr-cf)    Δ     old (cellina) new (cellina-cr-cf)    Δ     old (cellina) new (cellina-cr-cf)     Δ old (cellina) new (cellina-cr-cf)    Δ
sid     holdout_celltype                                                                                                                                                                     
crc_120 Fibroblast_CRC           0.712               0.712  0.0              0.24                0.24  0.0             7.201               7.279  0.08    178593.896          178593.896  0.0
crc_210 Fibroblast_CRC           0.935               0.935  0.0              0.32                0.32  0.0             7.624               7.585 -0.04     42593.299           42593.299  0.0
crc_221 Fibroblast_CRC           0.823               0.823  0.0              0.28                0.28  0.0             8.005               8.160  0.15     52082.906           52082.906  0.0
crc_231 Fibroblast_CRC           0.303               0.303  0.0              0.10                0.10  0.0             6.485               6.534  0.05     42320.260           42320.260  0.0
crc_232 Fibroblast_CRC           0.800               0.800  0.0              0.18                0.18  0.0             8.239               8.168 -0.07     28042.732           28042.732  0.0
crc_242 Fibroblast_CRC           0.895               0.895  0.0              0.32                0.32  0.0             6.694               6.861  0.17     32779.076           32779.076  0.0

In [7]:
comparisons['cellina-pert'].round(3)

pearson                                 direction_match_k                                 edistance_pca_log                                              rmse  \
                         old (cellina-pert) new (cellina-cr-cf-node)     Δ old (cellina-pert) new (cellina-cr-cf-node)     Δ old (cellina-pert) new (cellina-cr-cf-node)     Δ old (cellina-pert)   
sid     holdout_celltype                                                                                                                                                                            
crc_120 Fibroblast_CRC                0.907                    0.860 -0.05               0.36                     0.16 -0.20              7.262                    6.556 -0.71         178531.791   
crc_210 Fibroblast_CRC                0.953                    0.927 -0.03               0.24                     0.22 -0.02              8.727                    6.009 -2.72          42416.929   
crc_221 Fibroblast_CRC                0.924                    0.752 -0.17               0.38                     0.18 -0.20              9.624                    6.311 -3.31          51854.412   
crc_231 Fibroblast_CRC                0.317                    0.308 -0.01               0.10                     0.10  0.00              8.614                    6.619 -2.00          40811.416   
crc_232 Fibroblast_CRC                0.799                    0.888  0.09               0.16                     0.28  0.12              7.005                    8.297  1.29          28069.276   
crc_242 Fibroblast_CRC                0.896                    0.908  0.01               0.52                     0.42 -0.10              7.386                    6.291 -1.10          30867.358   

                                                            
                         new (cellina-cr-cf-node)        Δ  
sid     holdout_celltype                                    
crc_120 Fibroblast_CRC                 178736.581   204.79  
crc_210 Fibroblast_CRC                  42758.475   341.55  
crc_221 Fibroblast_CRC                  52244.024   389.61  
crc_231 Fibroblast_CRC                  42238.090  1426.67  
crc_232 Fibroblast_CRC                  27872.112  -197.16  
crc_242 Fibroblast_CRC                  32108.501  1241.14

## Mean over matched runs

Rows: metrics. Columns: old | new | Δ per pair.

In [8]:
summary = pd.concat({f'{o} → {n}': comparisons[o].mean().unstack().reindex(columns=comparisons[o].columns.get_level_values(1).unique())
                     .set_axis(['old', 'new', 'Δ'], axis=1)
                     for o, n in PAIRS.items()}, axis=1)
summary.round(3)

cellina → cellina-cr-cf                   cellina-pert → cellina-cr-cf-node                    
                                      old        new      Δ                               old        new        Δ
pearson                             0.745      0.745  0.000                             0.799      0.774   -0.027
direction_match_k                   0.240      0.240  0.000                             0.293      0.227   -0.067
edistance_pca_log                   7.375      7.431  0.057                             8.103      6.681   -1.425
rmse                            62735.362  62735.362  0.000                         62091.864  62659.631  567.767